In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Analysis_functions import *
import plotly.graph_objs as go
import statsmodels.api as sm

In [ ]:
# Read the CSV file
raw = np.loadtxt('Data/final.csv', delimiter=',', skiprows=1)

# Extract the relevant data
time = raw[:, 5]/1000
angle = raw[:, 4]
temp = np.delete(raw, [2, 3, 4, 5], axis=1)

# Calculate the average of the data
temp = np.mean(temp, axis=1)

N = temp.shape[0]
ambient = np.full(N, temp[0])

In [ ]:
# Plot the data
plt.plot(time, raw[:, 0], label='T1')
plt.plot(time, raw[:, 1], label='T2')
plt.plot(time, ambient, label='Ambient')

# Add labels and legend
plt.xlabel('X-axis Label')
plt.ylabel('Y-axis Label')
plt.legend()

#plt.xlim(0, 40)  # Adjust the range as needed
#plt.ylim(20, 30) 
plt.show()

In [ ]:
window = 20
N = temp.shape[0]
print(f'Number of data points: {N}')
frac = window / N  # Equivalent to MATLAB window parameter
print(f'Fraction: {frac}')

# Apply the LOWESS smoothing to the experimental temperature data
smoothed_temp = sm.nonparametric.lowess(temp, time, frac=frac)
# Apply the LOWESS smoothing to the ambient temperature data
smoothed_ambient = sm.nonparametric.lowess(ambient, time, frac=frac)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=time, y=temp, mode='lines', name='Temp'))
fig.add_trace(go.Scatter(x=smoothed_temp[:, 0], y=smoothed_temp[:, 1], mode='lines', name='Smoothed Temp'))
fig.add_trace(go.Scatter(x=time, y=ambient, mode='lines', name='Ambient'))
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=time, y=angle, mode='lines', name='Angle vs Time'))
fig.update_layout(title='Shader angle vs Time', xaxis_title='Time (s)', yaxis_title='Angle (Degrees)')
fig.show()

In [ ]:
mass_water = 105  # g
dT_threshold = 3

# Computes the power over the whole experiment
power_input = power(smoothed_temp[:, 0], smoothed_temp[:, 1], mass_water)
prw_vs_temp = np.array([(smoothed_temp[:, 1] - smoothed_ambient[:, 1])[1:], power_input])

# Sorts the data in increasing order of x values
sorted_indices = np.argsort(prw_vs_temp[0, :])
prw_vs_temp = prw_vs_temp[:, sorted_indices]

In [ ]:
# Performs a linear fit on the power vs temperature delta curve
prw_vs_temp_fit = sm.add_constant(prw_vs_temp)
X = np.column_stack((np.ones_like(prw_vs_temp_fit[0, :]), prw_vs_temp_fit[0, :]))
y = prw_vs_temp_fit[1, :]
model = sm.OLS(y, X)

results = model.fit()
# print(results.summary())
params = results.params
print(f'Slope of the fitted line: {params[1]} W/°C')

def linear_fit(x, params):
    return params[1]*x + params[0] 

fitted_line = linear_fit(prw_vs_temp[0, :], params)

In [ ]:
# Power vs temperature delta curve with the linear fit
fig = go.Figure()
fig.add_trace(go.Scatter(x=prw_vs_temp[0, :], y=prw_vs_temp[1, :], mode='lines', name='Data'))
fig.add_trace(go.Scatter(x=prw_vs_temp[0, :], y=fitted_line, mode='lines', name='fitted line'))
fig.show()

In [ ]:
# Computes the total energy harvested during the experiment
energy = energy_stored(temp, mass_water)
print(f'Energy harvested: {energy} J')